"""Computational search for aligned sets of odd primes.

This cell reproduces the two searches reported in Proposition 5.13
("Verification") of the article *Multiplicative-Order Parametrized Digraphs*.
Every routine that supports a claim in the article is paired with an
independent, deliberately naive routine in the verification cell, which
recomputes the same quantity by a different method.

Definitions
-----------
For distinct odd primes p_i, p_j write

    r_ij  =  ord_{p_j}(p_i),

the multiplicative order of p_i in (Z/p_j Z)^*.  A set P = {p_1, ..., p_n} of
n distinct odd primes is *aligned* when the n(n-1) orders r_ij are prime and
pairwise distinct.  Aligned sets of cardinality 2 exist, {3, 11} being the
smallest; none of cardinality 3 or more is known.

What the two searches certify
-----------------------------
Part A (`exhaustive_sweep`) decides the question inside an interval: it
examines every triple of odd primes below a bound X and reports whether any
is aligned.  It is complete below X and says nothing above it.

Part B (`confinement_search`) certifies an unbounded region.  It uses the
cyclotomic confinement equivalence of Theorem 5.10: for primes r, and primes
q not in {p, r},

    ord_q(p) = r    <=>    q | Phi_r(p) = (p^r - 1) / (p - 1).

Fixing p and an exact order r confines the modulus q to the prime divisors of
one explicit integer.  Factoring that integer yields the complete candidate
list, with no upper bound on q at all.

The quantity computed everywhere is not the multiplicative order itself but
the predicate "the order is prime, and if so which prime".  The following
elementary equivalence, proved in the article, makes that predicate cheap:

    ord_p(a) is a prime r   <=>   a != 1 (mod p)  and  a^r = 1 (mod p)
                                   for some prime r | p - 1,

and at most one prime r can satisfy the right-hand side.  Testing the
omega(p-1) prime divisors of p - 1 therefore settles the question, without
ever computing a full order.

Layout
------
    Section 1   arithmetic primitives
    Section 2   Part A, exhaustive sweep below a bound
    Section 3   Part B, cyclotomic confinement search
    Section 4   result records and persistence

The cell is deterministic: no randomness, no timing-dependent branching, no
parallelism.  Two runs with the same parameters produce identical output.
"""

In [1]:
from __future__ import annotations

import itertools
import json
import os
import time
from dataclasses import dataclass, field, asdict

import numpy as np
from sympy import factorint, primerange

__all__ = [
    "sieve_odd_primes",
    "smallest_prime_factor_sieve",
    "distinct_prime_factors",
    "prime_order",
    "prime_order_column",
    "prime_order_unbounded",
    "cyclotomic_value",
    "exhaustive_sweep",
    "confinement_candidates",
    "confinement_search",
    "SweepResult",
    "ConfinementResult",
    "append_run_log",
]


# =============================================================================
# Section 1.  Arithmetic primitives
# =============================================================================

def sieve_odd_primes(limit: int) -> list[int]:
    """Return the odd primes strictly below `limit`, ascending.

    The prime 2 is excluded throughout: every element of an aligned set is an
    odd prime by definition, since an order modulo 2 is always 1.
    """
    if limit <= 3:
        return []
    flags = bytearray([1]) * limit
    flags[0] = flags[1] = 0
    i = 2
    while i * i < limit:
        if flags[i]:
            flags[i * i::i] = bytearray(len(range(i * i, limit, i)))
        i += 1
    return [i for i in range(3, limit) if flags[i]]


def smallest_prime_factor_sieve(limit: int) -> np.ndarray:
    """Array `spf` with `spf[k]` the smallest prime factor of k, for k <= limit.

    Used to factor the numbers p - 1 for all p below the bound in one pass,
    which is far cheaper than factoring them one at a time.
    """
    spf = np.arange(limit + 1, dtype=np.int64)
    i = 2
    while i * i <= limit:
        if spf[i] == i:                      # i is prime
            block = spf[i * i::i]
            block[block == np.arange(i * i, limit + 1, i)] = i
        i += 1
    return spf


def distinct_prime_factors(n: int, spf: np.ndarray | None = None) -> list[int]:
    """Distinct prime divisors of `n`, ascending.

    Uses the sieve `spf` when `n` is within its range, and falls back to
    `sympy.factorint` otherwise, so the same routine serves both the bounded
    sweep of Part A and the unbounded candidates of Part B.
    """
    if spf is not None and n < len(spf):
        out = []
        while n > 1:
            p = int(spf[n])
            out.append(p)
            while n % p == 0:
                n //= p
        return out
    return sorted(factorint(n))


def prime_order(a: int, p: int, prime_divisors: list[int]) -> int | None:
    """Return r if `ord_p(a)` is the prime r, and None if the order is not prime.

    `prime_divisors` must be the list of distinct prime divisors of p - 1.

    Correctness.  If ord_p(a) = r with r prime, then r divides p - 1 by
    Lagrange, so r appears in the list, and a^r = 1.  Conversely, if a^r = 1
    for some prime r in the list, then ord_p(a) divides r, hence is 1 or r; the
    test `a != 1 (mod p)` excludes 1.  At most one r can pass: two of them
    would force the order to divide gcd(r, r') = 1.  The loop may therefore
    return on the first hit.

    This is the scalar reference implementation.  `prime_order_column` computes
    the same predicate for a whole array of bases at once; the two are checked
    against each other in the verification cell.
    """
    a %= p
    if a <= 1:                               # a = 0 cannot occur for distinct primes
        return None
    for r in prime_divisors:
        if pow(a, r, p) == 1:
            return r
    return None


def _powmod_array(bases: np.ndarray, exponent: int, modulus: int) -> np.ndarray:
    """Vectorized `bases ** exponent % modulus` by square-and-multiply.

    Requires `modulus` below 3 * 10^9 so that every intermediate product stays
    inside the signed 64-bit range; the caller asserts this.
    """
    result = np.ones_like(bases)
    base = bases % modulus
    e = exponent
    while e:
        if e & 1:
            result = result * base % modulus
        base = base * base % modulus
        e >>= 1
    return result


def prime_order_column(bases: np.ndarray, modulus: int,
                       prime_divisors: list[int],
                       check: bool = True) -> np.ndarray:
    """Prime orders of every entry of `bases` modulo `modulus`, as an array.

    Entry k of the result is r when `ord_modulus(bases[k])` is the prime r, and
    0 when the order is not prime (including when bases[k] = modulus, whose
    residue is 0, and when bases[k] = 1 mod modulus, whose order is 1).

    With `check=True` the routine asserts that no entry is ever assigned twice,
    which is the uniqueness argument given in the docstring of `prime_order`.
    """
    assert modulus < 3_000_000_000, "modulus too large for int64 intermediates"
    residues = bases % modulus
    eligible = residues > 1
    order = np.zeros(len(bases), dtype=np.int32)
    for r in prime_divisors:
        hit = eligible & (_powmod_array(residues, r, modulus) == 1)
        if check:
            assert not order[hit].any(), (
                f"two distinct prime orders modulo {modulus}: impossible")
        order[hit] = r
    return order


def cyclotomic_value(p: int, r: int) -> int:
    """Phi_r(p) for a prime r, that is (p^r - 1) / (p - 1) = 1 + p + ... + p^(r-1)."""
    return (p ** r - 1) // (p - 1)


# =============================================================================
# Section 2.  Part A -- exhaustive sweep below a bound
# =============================================================================
#
# Encoding of the table of orders.
# --------------------------------
# Let p_0 < p_1 < ... < p_{m-1} be the odd primes below the bound X.  The pair
# (base index b, modulus index c) with a prime order r = ord_{p_c}(p_b) is
# stored as the single integer
#
#       key  =  c * m + b            with the value r alongside,
#
# the modulus index leading.  The sweep processes one modulus c at a time in
# increasing order and, inside a modulus, the bases in increasing order, so the
# keys are produced already sorted and no sort is ever needed.  The reciprocal
# entry of `key` is `b * m + c`, located by binary search.
#
# At X = 3 * 10^5 the table holds about 4.2 * 10^7 entries, that is roughly
# 340 MB for the keys and 170 MB for the orders.  A dictionary of dictionaries
# would need tens of gigabytes for the same content.


@dataclass
class SweepResult:
    """Every quantity Part A contributes to Proposition 5.13(1)."""
    bound: int
    n_odd_primes: int
    n_ordered_pairs: int
    n_prime_order_pairs: int
    n_edges: int
    n_triangles: int
    n_aligned: int
    n_five_distinct: int
    collision_kinds_among_five_distinct: dict = field(default_factory=dict)
    aligned_sets: list = field(default_factory=list)
    seconds: float = 0.0

    def report(self) -> str:
        return "\n".join([
            f"Part A -- exhaustive sweep below {self.bound}",
            f"  odd primes                       {self.n_odd_primes:>15,}",
            f"  ordered pairs                    {self.n_ordered_pairs:>15,}",
            f"  pairs of prime order             {self.n_prime_order_pairs:>15,}",
            f"  edges of G                       {self.n_edges:>15,}",
            f"  triangles of G                   {self.n_triangles:>15,}",
            f"  aligned sets (6 distinct orders) {self.n_aligned:>15,}",
            f"  triples with 5 distinct orders   {self.n_five_distinct:>15,}",
            f"  collision kinds among those      "
            f"{self.collision_kinds_among_five_distinct}",
            f"  elapsed                          {self.seconds:>15.1f} s",
        ])


def _build_order_table(primes: list[int], checkpoint: str | None,
                       chunk: int, verbose: bool):
    """Table of all pairs of prime order among `primes`.

    Returns `(keys, orders)`, two parallel arrays sorted by key, with
    `key = modulus_index * m + base_index` as described above.

    A checkpoint is written every `chunk` moduli, so a run interrupted by a
    Colab disconnection resumes from the last completed modulus rather than
    from the start.  The checkpoint stores the number of primes it was built
    with, and a mismatch raises rather than silently reusing a stale file.
    """
    m = len(primes)
    bases = np.array(primes, dtype=np.int64)
    spf = smallest_prime_factor_sieve(primes[-1])
    factors = [distinct_prime_factors(p - 1, spf) for p in primes]
    del spf

    key_parts: list[np.ndarray] = []
    order_parts: list[np.ndarray] = []
    first_modulus = 0

    if checkpoint and os.path.exists(checkpoint):
        saved = np.load(checkpoint)
        if int(saved["m"]) != m:
            raise ValueError(
                f"checkpoint {checkpoint} was built with {int(saved['m'])} primes,"
                f" not {m}; delete it before rerunning")
        key_parts = [saved["keys"]]
        order_parts = [saved["orders"]]
        first_modulus = int(saved["next_modulus"])
        if verbose:
            print(f"  resuming from modulus index {first_modulus:,}"
                  f" ({len(key_parts[0]):,} entries already stored)")

    started = time.time()
    for c in range(first_modulus, m):
        order = prime_order_column(bases, primes[c], factors[c])
        hit = np.nonzero(order)[0]
        key_parts.append(c * m + hit.astype(np.int64))
        order_parts.append(order[hit])

        done = c + 1
        if done % chunk == 0 or done == m:
            key_parts = [np.concatenate(key_parts)]
            order_parts = [np.concatenate(order_parts)]
            if checkpoint:
                np.savez(checkpoint, keys=key_parts[0], orders=order_parts[0],
                         next_modulus=done, m=m)
            if verbose:
                elapsed = time.time() - started
                processed = done - first_modulus
                eta = elapsed * (m - done) / max(processed, 1)
                print(f"  modulus {done:>7,}/{m:,}   "
                      f"entries {len(key_parts[0]):>12,}   "
                      f"elapsed {elapsed:7.1f}s   eta {eta:7.1f}s", flush=True)

    if not key_parts:                        # nothing to do, m = 0
        return (np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int32))
    return key_parts[0], order_parts[0]


def _extract_edges(keys: np.ndarray, orders: np.ndarray, m: int,
                   block: int = 5_000_000):
    """Edges of the graph G of Proposition 5.13(1).

    Two odd primes p_u, p_v are adjacent in G when both ord_{p_v}(p_u) and
    ord_{p_u}(p_v) are prime and the two differ.  Returns `(u, v, r_uv, r_vu)`
    as four parallel arrays with u < v, where r_uv = ord_{p_v}(p_u).

    The table is scanned in blocks of `block` keys.  Every intermediate array
    is then of the size of one block rather than of the whole table, which
    keeps the peak footprint of this step below that of the table itself.
    """
    empty64 = np.empty(0, dtype=np.int64)
    if len(keys) == 0:
        empty32 = np.empty(0, dtype=orders.dtype)
        return empty64, empty64, empty32, empty32

    us, vs, forwards, backwards = [], [], [], []
    for start in range(0, len(keys), block):
        stop = min(start + block, len(keys))
        modulus_idx = keys[start:stop] // m
        base_idx = keys[start:stop] % m

        lower = base_idx < modulus_idx       # emit each edge once, as u < v
        modulus_idx = modulus_idx[lower]
        base_idx = base_idx[lower]
        r_forward = orders[start:stop][lower]

        reciprocal = base_idx * m + modulus_idx
        pos = np.searchsorted(keys, reciprocal)
        np.minimum(pos, len(keys) - 1, out=pos)
        present = keys[pos] == reciprocal
        r_backward = orders[pos]

        keep = present & (r_forward != r_backward)
        us.append(base_idx[keep])
        vs.append(modulus_idx[keep])
        forwards.append(r_forward[keep])
        backwards.append(r_backward[keep])

    return (np.concatenate(us), np.concatenate(vs),
            np.concatenate(forwards), np.concatenate(backwards))


def _classify_collision(pair_e, pair_f) -> str:
    """Shape of a repetition between two ordered pairs carrying equal orders.

    'column' -- the two pairs share their modulus, the second coordinate;
    'row'    -- they share their base, the first coordinate;
    'cross'  -- neither.

    Example 5.15 of the article reports that every repetition found among the
    triples with five distinct orders is of the 'column' shape.
    """
    if pair_e[1] == pair_f[1]:
        return "column"
    if pair_e[0] == pair_f[0]:
        return "row"
    return "cross"


def exhaustive_sweep(bound: int, outdir: str = "results", chunk: int = 500,
                     resume: bool = True, verbose: bool = True) -> SweepResult:
    """Decide the existence of aligned sets of cardinality 3 below `bound`.

    Complete inside the interval and silent outside it.  Writes to `outdir`:

        sweep_X<bound>.npz          checkpoint of the table of orders
        near_misses_X<bound>.csv    every triple with exactly 5 distinct orders
        aligned_X<bound>.json       every aligned set found, if any
        runs.jsonl                  one summary line per completed run

    The counts returned are exactly those quoted in Proposition 5.13(1).
    """
    os.makedirs(outdir, exist_ok=True)
    started = time.time()

    primes = sieve_odd_primes(bound)
    m = len(primes)
    if verbose:
        print(f"Part A -- {m:,} odd primes below {bound:,}"
              f" ({m * (m - 1):,} ordered pairs)", flush=True)

    if m == 0:
        return SweepResult(bound=bound, n_odd_primes=0, n_ordered_pairs=0,
                           n_prime_order_pairs=0, n_edges=0, n_triangles=0,
                           n_aligned=0, n_five_distinct=0,
                           seconds=round(time.time() - started, 1))

    checkpoint = os.path.join(outdir, f"sweep_X{bound}.npz") if resume else None
    keys, orders = _build_order_table(primes, checkpoint, chunk, verbose)
    n_prime_order_pairs = int(len(keys))

    u, v, r_uv, r_vu = _extract_edges(keys, orders, m)
    del keys, orders
    n_edges = int(len(u))
    if verbose:
        print(f"  {n_prime_order_pairs:,} pairs of prime order,"
              f" {n_edges:,} edges", flush=True)

    # Orders are needed only on edges: in a triangle all three pairs are edges,
    # so all six orders of a candidate triple are edge orders.
    order_of = {}
    adjacency: dict[int, set[int]] = {}
    for a, b, rab, rba in zip(u.tolist(), v.tolist(),
                              r_uv.tolist(), r_vu.tolist()):
        order_of[a * m + b] = rab            # ord_{p_b}(p_a)
        order_of[b * m + a] = rba
        adjacency.setdefault(a, set()).add(b)
        adjacency.setdefault(b, set()).add(a)
    del u, v, r_uv, r_vu

    near_path = os.path.join(outdir, f"near_misses_X{bound}.csv")
    aligned_path = os.path.join(outdir, f"aligned_X{bound}.json")

    n_triangles = n_aligned = n_five = 0
    kinds: dict[str, int] = {}
    aligned_sets: list = []

    with open(near_path, "w", newline="") as handle:
        handle.write("p1,p2,p3,r12,r13,r21,r23,r31,r32,"
                     "n_distinct,repeated_order,collision_kind\n")
        for a in sorted(adjacency):
            neighbours_a = adjacency[a]
            for b in sorted(x for x in neighbours_a if x > a):
                common = neighbours_a & adjacency[b]
                for c in sorted(x for x in common if x > b):
                    n_triangles += 1
                    triple = (a, b, c)
                    orders_of_triple = {
                        (x, y): order_of[x * m + y]
                        for x, y in itertools.permutations(triple, 2)
                    }
                    values = list(orders_of_triple.values())
                    n_distinct = len(set(values))

                    if n_distinct == 6:
                        n_aligned += 1
                        found = {
                            "primes": [primes[t] for t in triple],
                            "orders": {f"r_{x}{y}": r for (x, y), r
                                       in orders_of_triple.items()},
                        }
                        aligned_sets.append(found)
                        print("*** ALIGNED SET FOUND:", found, flush=True)
                        continue

                    if n_distinct != 5:
                        continue

                    n_five += 1
                    e, f = next(
                        (e, f) for e, f in itertools.combinations(
                            sorted(orders_of_triple), 2)
                        if orders_of_triple[e] == orders_of_triple[f]
                    )                        # exactly one repetition
                    kind = _classify_collision(e, f)
                    kinds[kind] = kinds.get(kind, 0) + 1
                    p1, p2, p3 = (primes[t] for t in triple)
                    row = [p1, p2, p3,
                           orders_of_triple[(a, b)], orders_of_triple[(a, c)],
                           orders_of_triple[(b, a)], orders_of_triple[(b, c)],
                           orders_of_triple[(c, a)], orders_of_triple[(c, b)],
                           n_distinct, orders_of_triple[e], kind]
                    handle.write(",".join(str(x) for x in row) + "\n")

    if aligned_sets:
        with open(aligned_path, "w") as handle:
            json.dump(aligned_sets, handle, indent=2)

    result = SweepResult(
        bound=bound,
        n_odd_primes=m,
        n_ordered_pairs=m * (m - 1),
        n_prime_order_pairs=n_prime_order_pairs,
        n_edges=n_edges,
        n_triangles=n_triangles,
        n_aligned=n_aligned,
        n_five_distinct=n_five,
        collision_kinds_among_five_distinct=kinds,
        aligned_sets=aligned_sets,
        seconds=round(time.time() - started, 1),
    )
    append_run_log(outdir, "A", asdict(result))
    if verbose:
        print(result.report(), flush=True)
    return result


# =============================================================================
# Section 3.  Part B -- cyclotomic confinement search
# =============================================================================

def _trial_division(n: int, limit: int):
    """Split n as (known prime factors below `limit`, unfactored cofactor)."""
    known: list[int] = []
    cofactor = n
    for q in primerange(2, limit):
        if q * q > cofactor:
            break
        if cofactor % q == 0:
            known.append(q)
            while cofactor % q == 0:
                cofactor //= q
    return known, cofactor


def prime_order_unbounded(a: int, p: int, trial_limit: int = 10 ** 6,
                          allow_hard_factoring: bool = True):
    """Prime order of `a` modulo `p` when p - 1 may be too large to factor.

    Returns `(r, resolved)`: `r` is the prime order, or None when the order is
    not prime; `resolved` is False only when the routine gave up, which can
    happen only if `allow_hard_factoring` is False.

    Method.  Write p - 1 = K * C with K the part factored by trial division
    below `trial_limit` and C the unfactored cofactor, so every prime divisor
    of C exceeds `trial_limit`.  Let d = ord_p(a).

      * If a^K = 1 then d divides K, so a prime d is one of the known primes,
        and testing them settles the question.
      * Otherwise d does not divide K.  If in addition a^C != 1 then d does
        not divide C either; were d a prime it would divide p - 1 = K * C and
        hence divide K or C, a contradiction.  So d is not prime, decided
        without factoring C at all.
      * Only when a^K != 1 and a^C = 1 does d divide C, forcing a prime d to
        be a prime divisor of C.  This is the one case in which C must be
        factored, and it is rare.

    The three cases are exhaustive.
    """
    a %= p
    if a <= 1:
        return None, True

    known, cofactor = _trial_division(p - 1, trial_limit)
    K = (p - 1) // cofactor

    if pow(a, K, p) == 1:
        for r in known:
            if pow(a, r, p) == 1:
                return r, True
        return None, True

    if pow(a, cofactor, p) != 1:
        return None, True                    # order provably not prime

    if not allow_hard_factoring:
        return None, False
    for r in sorted(factorint(cofactor)):
        if pow(a, r, p) == 1:
            return r, True
    return None, True


def confinement_candidates(p: int, r: int, digit_cap: int = 50,
                           verbose: bool = False):
    """Complete list of odd primes q with `ord_q(p) = r`, or None if skipped.

    By Theorem 5.10 such a q is exactly a prime divisor of Phi_r(p) other than
    p and r, so factoring that single integer produces the whole list, with no
    upper bound on q.  The divisor 2 is dropped as well, since the elements of
    an aligned set are odd primes; it can occur only for r = 2, where Phi_2(p)
    = p + 1 is even.  None is returned when Phi_r(p) exceeds `digit_cap`
    decimal digits and was therefore not factored; callers must report those
    values of r alongside any completeness claim.
    """
    value = cyclotomic_value(p, r)
    if len(str(value)) > digit_cap:
        if verbose:
            print(f"    r = {r:>3}: Phi_r({p}) has {len(str(value))} digits,"
                  f" above the cap of {digit_cap}; skipped")
        return None
    candidates = sorted(q for q in factorint(value) if q not in (2, p, r))
    if verbose:
        print(f"    r = {r:>3}: Phi_r({p}) has {len(str(value)):>3} digits,"
              f" {len(candidates):>3} admissible prime divisors")
    return candidates


@dataclass
class ConfinementResult:
    """Every quantity Part B contributes to Proposition 5.13(2)."""
    p_max: int
    r_max: int
    digit_cap: int
    allow_order_equal_to_base: bool
    allow_hard_factoring: bool
    n_base_primes: int
    n_candidate_pairs: int
    n_aligned: int
    n_undecided: int
    skipped: dict = field(default_factory=dict)
    aligned_sets: list = field(default_factory=list)
    seconds: float = 0.0

    def report(self) -> str:
        return "\n".join([
            f"Part B -- cyclotomic confinement, p < {self.p_max},"
            f" r <= {self.r_max}",
            f"  base primes p                    {self.n_base_primes:>15,}",
            f"  candidate pairs (p', p'')        {self.n_candidate_pairs:>15,}",
            f"  aligned sets                     {self.n_aligned:>15,}",
            f"  pairs left undecided             {self.n_undecided:>15,}",
            f"  hard factoring allowed           "
            f"{str(self.allow_hard_factoring):>15}",
            f"  orders skipped (digit cap)       {self.skipped}",
            f"  elapsed                          {self.seconds:>15.1f} s",
        ])


def confinement_search(p_max: int = 100, r_max: int = 23, digit_cap: int = 50,
                       allow_order_equal_to_base: bool = True,
                       allow_hard_factoring: bool = True,
                       trial_limit: int = 10 ** 6, outdir: str = "results",
                       verbose: bool = True) -> ConfinementResult:
    """Search for aligned sets {p, p', p''} with p < `p_max` and no bound on p', p''.

    For every odd prime p below `p_max` and every unordered choice of two
    distinct primes r < r' at most `r_max`, the complete candidate lists for
    p' and p'' are obtained by factoring Phi_r(p) and Phi_{r'}(p).  Every
    resulting pair is then completed: the remaining four orders are computed
    and the six are tested for being prime and pairwise distinct.  What is
    decided is therefore the following statement: no aligned triple contains a
    prime p below `p_max` whose two outgoing orders are both at most `r_max`.

    `allow_order_equal_to_base` controls whether the value r = p is admitted as
    a prescribed order.  It is arithmetically legitimate -- ord_q(p) = p simply
    forces p | q - 1 -- and it affects the reported pair count, so it is made
    explicit rather than silently fixed.

    `allow_hard_factoring` is passed to `prime_order_unbounded`.  With the
    default True the routine factors whatever it must and `n_undecided` is
    zero by construction; set it to False to bound the work per pair, in which
    case `n_undecided` counts the pairs that were left open and a completeness
    claim requires it to be zero.
    """
    os.makedirs(outdir, exist_ok=True)
    started = time.time()

    base_primes = sieve_odd_primes(p_max)
    order_values = list(primerange(2, r_max + 1))

    n_pairs = n_aligned = n_undecided = 0
    skipped: dict[int, list[int]] = {}
    aligned_sets: list = []

    for p1 in base_primes:
        if verbose:
            print(f"  base prime p = {p1}", flush=True)
        candidates: dict[int, list[int]] = {}
        for r in order_values:
            if r == p1 and not allow_order_equal_to_base:
                continue
            found = confinement_candidates(p1, r, digit_cap, verbose)
            if found is None:
                skipped.setdefault(p1, []).append(r)
            else:
                candidates[r] = found

        factors_of_p1 = distinct_prime_factors(p1 - 1)

        # The candidate lists are pairwise disjoint and exclude p1 itself, so
        # p1, p2 and p3 are automatically three distinct primes.
        for r2, r3 in itertools.combinations(sorted(candidates), 2):
            for p2 in candidates[r2]:
                for p3 in candidates[r3]:
                    n_pairs += 1
                    used = {r2, r3}

                    # Cheap tests first: orders modulo the small prime p1.
                    r21 = prime_order(p2, p1, factors_of_p1)
                    if r21 is None or r21 in used:
                        continue
                    used.add(r21)
                    r31 = prime_order(p3, p1, factors_of_p1)
                    if r31 is None or r31 in used:
                        continue
                    used.add(r31)

                    # Expensive tests: orders modulo the large candidates.
                    r23, ok = prime_order_unbounded(
                        p2, p3, trial_limit, allow_hard_factoring)
                    if not ok:
                        n_undecided += 1
                        continue
                    if r23 is None or r23 in used:
                        continue
                    used.add(r23)
                    r32, ok = prime_order_unbounded(
                        p3, p2, trial_limit, allow_hard_factoring)
                    if not ok:
                        n_undecided += 1
                        continue
                    if r32 is None or r32 in used:
                        continue

                    orders = {(1, 2): r2, (1, 3): r3, (2, 1): r21,
                              (3, 1): r31, (2, 3): r23, (3, 2): r32}
                    assert len(set(orders.values())) == 6, orders
                    found = {"primes": [p1, p2, p3],
                             "orders": {f"r_{x}{y}": v
                                        for (x, y), v in orders.items()}}
                    aligned_sets.append(found)
                    n_aligned += 1
                    print("*** ALIGNED SET FOUND:", found, flush=True)

    result = ConfinementResult(
        p_max=p_max, r_max=r_max, digit_cap=digit_cap,
        allow_order_equal_to_base=allow_order_equal_to_base,
        allow_hard_factoring=allow_hard_factoring,
        n_base_primes=len(base_primes),
        n_candidate_pairs=n_pairs,
        n_aligned=n_aligned,
        n_undecided=n_undecided,
        skipped={str(k): v for k, v in skipped.items()},
        aligned_sets=aligned_sets,
        seconds=round(time.time() - started, 1),
    )
    append_run_log(outdir, "B", asdict(result))
    if verbose:
        print(result.report(), flush=True)
    return result


# =============================================================================
# Section 4.  Persistence
# =============================================================================

def append_run_log(outdir: str, part: str, payload: dict) -> None:
    """Append one JSON line per completed run to `outdir/runs.jsonl`."""
    os.makedirs(outdir, exist_ok=True)
    record = {"part": part, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
              **payload}
    with open(os.path.join(outdir, "runs.jsonl"), "a") as handle:
        handle.write(json.dumps(record, default=str) + "\n")

Independent checks on the searches of the previous cell.

Nothing here is used by the searches themselves.  Every routine recomputes a
quantity that the searches also compute, by a different method -- in most
cases `sympy.n_order`, which computes a full multiplicative order and shares
no code with the prime-order predicate used in the searches.

This is the independent routine referred to at the end of the proof of
Proposition 5.13.  Running `check_all()` executes the whole battery and
returns the list of discrepancies, which must be empty.


In [2]:
from __future__ import annotations

import itertools

import numpy as np
from sympy import factorint, isprime, n_order, primerange


# -----------------------------------------------------------------------------
# 1.  The prime-order predicate against a full order computation
# -----------------------------------------------------------------------------

def check_prime_order_against_sympy(modulus_bound: int = 2000,
                                    base_bound: int = 400) -> list:
    """`prime_order` agrees with `sympy.n_order` on a dense range of inputs.

    For every odd prime p below `modulus_bound` and every base a below
    `base_bound`, the predicate must return the true order when that order is
    prime, and None otherwise.
    """
    problems = []
    spf = smallest_prime_factor_sieve(modulus_bound)
    for p in sieve_odd_primes(modulus_bound):
        factors = distinct_prime_factors(p - 1, spf)
        for a in range(2, min(p, base_bound)):
            reported = prime_order(a, p, factors)
            true_order = n_order(a, p)
            if reported is None:
                if isprime(true_order):
                    problems.append(("missed prime order", a, p, true_order))
            elif reported != true_order:
                problems.append(("wrong order", a, p, reported, true_order))
    return problems


def check_vectorized_against_scalar(bound: int = 3000) -> list:
    """`prime_order_column` agrees with `prime_order` entry by entry.

    The sweep of Part A uses only the vectorized routine, so this is the check
    that connects it to the scalar reference validated above.
    """
    problems = []
    primes = sieve_odd_primes(bound)
    bases = np.array(primes, dtype=np.int64)
    spf = smallest_prime_factor_sieve(bound)
    for modulus in primes:
        factors = distinct_prime_factors(modulus - 1, spf)
        column = prime_order_column(bases, modulus, factors)
        for index, base in enumerate(primes):
            expected = prime_order(base, modulus, factors) or 0
            if int(column[index]) != expected:
                problems.append(("column mismatch", base, modulus,
                                 int(column[index]), expected))
    return problems


def check_unbounded_prime_order(modulus_bound: int = 5000,
                                base_bound: int = 200) -> list:
    """`prime_order_unbounded` agrees with `sympy.n_order`.

    The routine of Part B decides primality of the order from a partial
    factorization of p - 1; a deliberately small `trial_limit` is used here so
    that all three of its branches are exercised rather than only the first.
    Each input is run twice, once with hard factoring allowed and once with it
    forbidden, so that the `resolved` flag is exercised too: the cautious call
    may give up, but whenever it answers it must give the same answer.
    """
    problems = []
    for p in sieve_odd_primes(modulus_bound):
        for a in range(2, min(p, base_bound)):
            reported, resolved = prime_order_unbounded(a, p, trial_limit=30)
            if not resolved:
                problems.append(("unresolved with hard factoring allowed", a, p))
                continue
            true_order = n_order(a, p)
            if reported is None:
                if isprime(true_order):
                    problems.append(("missed prime order", a, p, true_order))
            elif reported != true_order:
                problems.append(("wrong order", a, p, reported, true_order))

            cautious, resolved = prime_order_unbounded(
                a, p, trial_limit=30, allow_hard_factoring=False)
            if resolved and cautious != reported:
                problems.append(("cautious branch disagrees", a, p,
                                 cautious, reported))
    return problems


# -----------------------------------------------------------------------------
# 2.  The cyclotomic confinement equivalence
# -----------------------------------------------------------------------------

def check_confinement_equivalence(base_primes=(3, 5, 7, 11, 13, 101),
                                  order_values=(2, 3, 5, 7, 11),
                                  modulus_bound: int = 50_000) -> list:
    """Both directions of Theorem 5.10, by brute force.

    Soundness: every prime divisor of Phi_r(p) other than p and r has order
    exactly r modulo itself.  Completeness: every prime q below
    `modulus_bound` with ord_q(p) = r divides Phi_r(p).  Completeness is the
    direction Part B relies on, since it is what makes the candidate list
    exhaustive without any bound on q.
    """
    problems = []
    for p in base_primes:
        for r in order_values:
            divisors = {q for q in factorint(cyclotomic_value(p, r))
                        if q not in (2, p, r)}
            for q in divisors:
                if n_order(p, q) != r:
                    problems.append(("not of order r", p, r, q, n_order(p, q)))
            for q in primerange(3, modulus_bound):
                if q in (p, r):
                    continue
                if n_order(p, q) == r and q not in divisors:
                    problems.append(("candidate list incomplete", p, r, q))
    return problems


# -----------------------------------------------------------------------------
# 3.  The sweep against a brute-force reference
# -----------------------------------------------------------------------------

def reference_sweep(bound: int) -> dict:
    """Part A recomputed in the most obvious possible way.

    Every ordered pair is handled with `sympy.n_order` and every triple with
    three nested loops, so the cost is quadratic in the number of primes for
    the table and cubic for the triangles.  That confines it to bounds of a
    few thousand.
    """
    primes = sieve_odd_primes(bound)
    m = len(primes)
    orders: dict[tuple[int, int], int] = {}
    n_prime_order_pairs = 0
    for i, base in enumerate(primes):
        for j, modulus in enumerate(primes):
            if i == j:
                continue
            value = n_order(base, modulus)
            if isprime(value):
                orders[(i, j)] = value
                n_prime_order_pairs += 1

    edges = set()
    for i, j in itertools.combinations(range(m), 2):
        if (i, j) in orders and (j, i) in orders and orders[(i, j)] != orders[(j, i)]:
            edges.add((i, j))

    n_triangles = n_aligned = n_five = 0
    kinds: dict[str, int] = {}
    for i, j, k in itertools.combinations(range(m), 3):
        if (i, j) not in edges or (i, k) not in edges or (j, k) not in edges:
            continue
        n_triangles += 1
        values = {(x, y): orders[(x, y)]
                  for x, y in itertools.permutations((i, j, k), 2)}
        distinct = len(set(values.values()))
        if distinct == 6:
            n_aligned += 1
        elif distinct == 5:
            n_five += 1
            for e, f in itertools.combinations(sorted(values), 2):
                if values[e] == values[f]:
                    kind = ("column" if e[1] == f[1]
                            else "row" if e[0] == f[0] else "cross")
                    kinds[kind] = kinds.get(kind, 0) + 1
                    break
    return {
        "n_odd_primes": m,
        "n_prime_order_pairs": n_prime_order_pairs,
        "n_edges": len(edges),
        "n_triangles": n_triangles,
        "n_aligned": n_aligned,
        "n_five_distinct": n_five,
        "collision_kinds_among_five_distinct": kinds,
    }


def check_sweep_against_reference(bound: int = 3000, outdir: str = "results",
                                  verbose: bool = True) -> list:
    """The fast sweep and the brute-force reference return the same counts."""
    fast = exhaustive_sweep(bound, outdir=outdir, resume=False, verbose=False)
    slow = reference_sweep(bound)
    problems = []
    for field_name, expected in slow.items():
        obtained = getattr(fast, field_name)
        if obtained != expected:
            problems.append((field_name, obtained, expected))
    if verbose:
        print(f"  sweep below {bound}: fast {fast.n_triangles:,} triangles,"
              f" reference {slow['n_triangles']:,}")
    return problems


# -----------------------------------------------------------------------------
# 4.  Fixed witnesses quoted in the article
# -----------------------------------------------------------------------------

def check_known_witnesses() -> list:
    """The two explicit examples of the article, recomputed from scratch."""
    problems = []

    # Example 5.12: {3, 11} is an aligned set of cardinality 2.
    if (n_order(3, 11), n_order(11, 3)) != (5, 2):
        problems.append(("aligned pair {3,11}",
                         n_order(3, 11), n_order(11, 3)))

    # Example 5.15: {11, 43, 16333} has six prime orders, five of them distinct,
    # the repetition being r_13 = r_23, two orders modulo the same element.
    p1, p2, p3 = 11, 43, 16333
    expected = {"r12": 7, "r13": 1361, "r21": 2,
                "r23": 1361, "r31": 5, "r32": 3}
    obtained = {"r12": n_order(p1, p2), "r13": n_order(p1, p3),
                "r21": n_order(p2, p1), "r23": n_order(p2, p3),
                "r31": n_order(p3, p1), "r32": n_order(p3, p2)}
    if obtained != expected:
        problems.append(("near miss {11,43,16333}", obtained, expected))
    if sorted(factorint(p3 - 1).items()) != [(2, 2), (3, 1), (1361, 1)]:
        problems.append(("factorization of 16333 - 1",
                         factorint(p3 - 1)))
    return problems


def recheck_triples(triples) -> list:
    """Recompute the six orders of each triple with `sympy.n_order`.

    Applied to the triples exported by a search, this is the check that no
    reported order is an artefact of the fast predicate.
    """
    problems = []
    for triple in triples:
        for a, b in itertools.permutations(triple, 2):
            value = n_order(a, b)
            if not isprime(value):
                problems.append(("reported order not prime", a, b, value))
    return problems


# -----------------------------------------------------------------------------

def check_all(sweep_bound: int = 3000, verbose: bool = True) -> list:
    """Run every check and return the combined list of discrepancies."""
    battery = [
        ("prime_order vs sympy.n_order", check_prime_order_against_sympy),
        ("vectorized vs scalar", check_vectorized_against_scalar),
        ("unbounded prime order vs sympy", check_unbounded_prime_order),
        ("cyclotomic confinement equivalence", check_confinement_equivalence),
        ("known witnesses", check_known_witnesses),
        ("sweep vs brute-force reference",
         lambda: check_sweep_against_reference(sweep_bound, verbose=verbose)),
    ]
    all_problems = []
    for name, routine in battery:
        problems = routine()
        all_problems.extend(problems)
        if verbose:
            status = "ok" if not problems else f"{len(problems)} PROBLEMS"
            print(f"  [{status:>12}]  {name}", flush=True)
            for item in problems[:5]:
                print("      ", item)
    if verbose:
        print("verification:",
              "clean" if not all_problems
              else f"{len(all_problems)} discrepancies")
    return all_problems

Exploratory search for aligned sets of cardinality 3 above the sweep bound.

This cell supports no claim in the article.  Part A of the search cell is
complete below a bound and Part B is complete for prescribed small orders; the
search here is neither, and its negative outcome certifies nothing.

An anchor pair (p_1, p_2) whose two mutual orders r_12 and r_21 are already
prime and distinct is fixed, and a large p_3 is scanned over blocks.  The four
orders still to be tested split into two groups of very different cost:

    r_31 = ord_{p_1}(p_3)  and  r_32 = ord_{p_2}(p_3)

depend on p_3 only through its residue modulo the small primes p_1 and p_2, so
they are read off a precomputed table in constant time;

    r_13 = ord_{p_3}(p_1)  and  r_23 = ord_{p_3}(p_2)

require the factorization of p_3 - 1, which is the expensive step.  Scanning in
that order means the expensive step runs only on the few p_3 that already pass
the cheap one.  Before either group, the necessary condition on p_3 - 1 derived
in the markdown cell above removes most primes at almost no cost.

Anchors are ranked by the density of residues that would survive the cheap
filter at p_1 and at p_2 separately.  The ranking ignores the further exclusion
r_32 != r_31, so it is an upper bound on the true survival probability rather
than the probability itself.


In [3]:
from __future__ import annotations

import itertools
import time

from sympy import factorint, isprime


SMALL_PRIME_LIMIT = 100_000
_SMALL_PRIMES = [2] + sieve_odd_primes(SMALL_PRIME_LIMIT)


# -----------------------------------------------------------------------------
# Sieving
# -----------------------------------------------------------------------------

def segmented_sieve(low: int, high: int) -> list[int]:
    """Primes in the interval [low, high), without allocating a sieve of size high.

    Sieving by the primes of `_SMALL_PRIMES` up to sqrt(high); the caller is
    responsible for keeping high below SMALL_PRIME_LIMIT squared.
    """
    if low < 2:
        low = 2
    if high <= low:
        return []
    assert high <= SMALL_PRIME_LIMIT ** 2, "block above the reach of _SMALL_PRIMES"
    size = high - low
    flags = bytearray([1]) * size
    for base in _SMALL_PRIMES:
        if base * base >= high:
            break
        start = max(base * base, ((low + base - 1) // base) * base)
        if start >= high:
            continue
        flags[start - low::base] = bytearray(len(range(start, high, base)))
    return [low + i for i in range(size) if flags[i]]


def has_two_distinct_odd_prime_factors(n: int, limit: int = 2000) -> bool:
    """Necessary condition on p_3 - 1, decided by trial division below `limit`.

    Returns False only when n is proved to carry fewer than two distinct odd
    prime factors, so no valid p_3 is ever discarded.  A remainder that resists
    trial division is kept unless it is prime: it may be a prime power, in
    which case the triple is rejected later at full cost.
    """
    while n % 2 == 0:
        n //= 2
    count = 0
    for q in _SMALL_PRIMES:
        if q > limit or q * q > n:
            break
        if n % q == 0:
            count += 1
            if count >= 2:
                return True
            while n % q == 0:
                n //= q
    if n == 1:
        return count >= 2
    if count >= 1:
        return True                          # the remainder is coprime to it
    return not isprime(n)


# -----------------------------------------------------------------------------
# Constant-time prime-order lookup modulo a small prime
# -----------------------------------------------------------------------------

def residue_table(p: int) -> list[int]:
    """Table T of length p with T[c] = r when ord_p(c) is the prime r, else 0.

    Built by generating the elements of each prime order directly: for a prime
    r dividing p - 1, an element h of order r is obtained as g^((p-1)/r) for
    the first g with that power different from 1, and its powers h, ..., h^(r-1)
    are exactly the elements of order r.  This touches only the elements of
    prime order rather than all p - 1 residues.
    """
    table = [0] * p
    for r in sorted(factorint(p - 1)):
        generator = None
        for g in range(2, p):
            candidate = pow(g, (p - 1) // r, p)
            if candidate != 1:
                generator = candidate
                break
        if generator is None:
            continue
        x = generator
        for _ in range(r - 1):
            table[x] = r
            x = x * generator % p
    return table


# -----------------------------------------------------------------------------
# Anchors
# -----------------------------------------------------------------------------

def build_anchors(bound: int, how_many: int) -> list[dict]:
    """Anchor pairs (p_1, p_2) ranked by surviving residue density.

    An anchor needs two mutual orders that are prime and distinct.  Its weight
    is the product over the two columns of the proportion of residues whose
    order is a prime not already used.  A weight of zero identifies a
    degenerate anchor -- p_1 = 3, for instance, where the only available prime
    order is 2 and it is already consumed.
    """
    primes = sieve_odd_primes(bound)
    factors = {p: sorted(factorint(p - 1)) for p in primes}
    anchors = []
    for p1, p2 in itertools.combinations(primes, 2):
        r12 = prime_order(p1, p2, factors[p2])
        if r12 is None:
            continue
        r21 = prime_order(p2, p1, factors[p1])
        if r21 is None or r21 == r12:
            continue
        used = (r12, r21)
        w1 = sum(r - 1 for r in factors[p1] if r not in used) / (p1 - 1)
        w2 = sum(r - 1 for r in factors[p2] if r not in used) / (p2 - 1)
        if w1 <= 0 or w2 <= 0:
            continue
        anchors.append({"weight": w1 * w2, "p1": p1, "p2": p2,
                        "r12": r12, "r21": r21})
    anchors.sort(key=lambda item: -item["weight"])
    return anchors[:how_many]


# -----------------------------------------------------------------------------
# The scan
# -----------------------------------------------------------------------------

def directed_search(anchors: list[dict], start: int, block: int = 5_000_000,
                    report_every: float = 30.0, stop_at_first: bool = True):
    """Scan p_3 over consecutive blocks against a fixed list of anchors.

    Runs until an aligned set is found, or indefinitely; interrupt it with the
    Colab stop button.  Every triple with five of the six orders distinct is
    printed as it appears.
    """
    tables = {}
    for anchor in anchors:
        for p in (anchor["p1"], anchor["p2"]):
            if p not in tables:
                tables[p] = residue_table(p)

    low = start
    n_cheap = n_full = n_near = n_skipped = 0
    started = last_report = time.time()

    while True:
        high = low + block
        block_primes = segmented_sieve(low, high)
        print(f"  block [{low:,}, {high:,})  {len(block_primes):,} primes"
              f"  ({time.time() - started:.0f}s)", flush=True)

        for p3 in block_primes:
            if not has_two_distinct_odd_prime_factors(p3 - 1):
                n_skipped += 1
                continue
            factors_of_p3 = None

            for anchor in anchors:
                p1, p2 = anchor["p1"], anchor["p2"]
                r12, r21 = anchor["r12"], anchor["r21"]

                r31 = tables[p1][p3 % p1]
                if r31 == 0 or r31 in (r12, r21):
                    continue
                r32 = tables[p2][p3 % p2]
                if r32 == 0 or r32 in (r12, r21, r31):
                    continue
                n_cheap += 1

                if factors_of_p3 is None:
                    factors_of_p3 = sorted(factorint(p3 - 1))
                r13 = prime_order(p1, p3, factors_of_p3)
                if r13 is None or r13 in (r12, r21, r31, r32):
                    continue
                r23 = prime_order(p2, p3, factors_of_p3)
                if r23 is None:
                    continue
                n_full += 1

                orders = (r12, r21, r13, r31, r23, r32)
                distinct = len(set(orders))
                if distinct == 6:
                    print(f"*** ALIGNED SET FOUND: {{{p1}, {p2}, {p3}}}"
                          f"  orders r12={r12} r21={r21} r13={r13}"
                          f" r31={r31} r23={r23} r32={r32}", flush=True)
                    if stop_at_first:
                        return (p1, p2, p3, orders)
                elif distinct == 5:
                    n_near += 1
                    print(f"    near miss {{{p1}, {p2}, {p3}}}  {orders}",
                          flush=True)

            if time.time() - last_report > report_every:
                print(f"    p3 ~ {p3:,}   cheap {n_cheap:,}   full {n_full:,}"
                      f"   near {n_near:,}   skipped {n_skipped:,}"
                      f"   {time.time() - started:.0f}s", flush=True)
                last_report = time.time()

        low = high

---
## 1. Independent verification

Every quantity the searches compute is recomputed here by a different method, in most
cases `sympy.n_order`, which computes a full multiplicative order and shares no code with
the prime-order predicate used in the searches. The battery covers:

1. the scalar predicate `prime_order` against `sympy.n_order` on a dense range of inputs;
2. the vectorized `prime_order_column`, used by Part A, against that scalar predicate;
3. `prime_order_unbounded`, used by Part B, against `sympy.n_order`, with a deliberately
   small trial-division limit so that all three of its branches are exercised;
4. both directions of the cyclotomic confinement equivalence of Theorem 5.10, by brute
   force -- in particular the completeness direction, which is what makes the candidate
   lists of Part B exhaustive with no upper bound;
5. the two explicit examples of the article, $\{3,11\}$ and $\{11,43,16333\}$;
6. the whole of Part A below a small bound against a transparent reference implementation.

The assertion below must pass.

In [4]:
problems = check_all(sweep_bound=3000)
assert not problems, problems

  [          ok]  prime_order vs sympy.n_order
  [          ok]  vectorized vs scalar
  [          ok]  unbounded prime order vs sympy
  [          ok]  cyclotomic confinement equivalence
  [          ok]  known witnesses
  sweep below 3000: fast 345 triangles, reference 345
  [          ok]  sweep vs brute-force reference
verification: clean


---
## 2. The predicate that makes the search cheap

The searches never compute a multiplicative order. They compute the predicate *"the order
is prime, and if so which prime"*, which is settled by the following elementary
equivalence, proved in the article:

$$\operatorname{ord}_p(a) \text{ is a prime } r \iff a \not\equiv 1 \pmod p \text{ and } a^r \equiv 1 \pmod p \text{ for some prime } r \mid p-1,$$

and at most one prime $r$ can satisfy the right-hand side. Testing the $\omega(p-1)$ prime
divisors of $p-1$ therefore settles the question, and no full order is ever computed. The
vectorized implementation carries an assertion that no entry is ever assigned twice, which
is the computational counterpart of that uniqueness claim.

In [5]:
from sympy import factorint

p = 16333
print("16333 - 1 =", factorint(p - 1))
for a in (11, 43, 2):
    print(f"  ord_{p}({a:>3}) prime? -> {prime_order(a, p, distinct_prime_factors(p - 1))}")

16333 - 1 = {2: 2, 3: 1, 1361: 1}
  ord_16333( 11) prime? -> 1361
  ord_16333( 43) prime? -> 1361
  ord_16333(  2) prime? -> None


---
## 3. Part A -- exhaustive sweep

Let $G$ be the graph whose vertices are the odd primes below $X$, with $p$ and $p'$
adjacent when $\operatorname{ord}_{p'}(p)$ and $\operatorname{ord}_p(p')$ are both prime
and distinct from each other. An aligned set of cardinality $3$ is a triangle of $G$ whose
six orders are pairwise distinct, so it suffices to enumerate the triangles.

The table of orders is held as one flat sorted array of keys; the encoding and its cost
are described in the comment opening Section 2 of the search cell. At $X = 3\cdot10^5$ it
occupies about 510 MB.

Start small to see the shape of the output, then run the published bound.

In [ ]:
small = exhaustive_sweep(20_000)

Part A -- 2,261 odd primes below 20,000 (5,109,860 ordered pairs)
  resuming from modulus index 2,261 (439,700 entries already stored)
  439,700 pairs of prime order, 18,296 edges
Part A -- exhaustive sweep below 20000
  odd primes                                 2,261
  ordered pairs                          5,109,860
  pairs of prime order                     439,700
  edges of G                                18,296
  triangles of G                            16,366
  aligned sets (6 distinct orders)               0
  triples with 5 distinct orders                 5
  collision kinds among those      {'column': 5}
  elapsed                                      0.3 s


In [6]:
# Published bound. About 7 minutes on one core; checkpointed every 4000 moduli,
# so a Colab disconnection costs only the current chunk.
result = exhaustive_sweep(300_000, chunk=4000)

Part A -- 25,996 odd primes below 300,000 (675,766,020 ordered pairs)
  modulus   4,000/25,996   entries    8,360,692   elapsed    49.0s   eta   269.6s
  modulus   8,000/25,996   entries   15,112,628   elapsed   101.9s   eta   229.2s
  modulus  12,000/25,996   entries   21,407,330   elapsed   159.5s   eta   186.0s
  modulus  16,000/25,996   entries   27,700,374   elapsed   218.2s   eta   136.3s
  modulus  20,000/25,996   entries   33,692,551   elapsed   284.4s   eta    85.3s
  modulus  24,000/25,996   entries   39,549,440   elapsed   345.9s   eta    28.8s
  modulus  25,996/25,996   entries   42,465,610   elapsed   378.2s   eta     0.0s
  42,465,610 pairs of prime order, 1,237,384 edges
Part A -- exhaustive sweep below 300000
  odd primes                                25,996
  ordered pairs                        675,766,020
  pairs of prime order                  42,465,610
  edges of G                             1,237,384
  triangles of G                         8,877,957
  aligned 

### Near misses

Every triple with exactly five distinct orders is exported. The `collision_kind` column
records the shape of the repetition: `column` when the two equal orders share their
modulus, `row` when they share their base, `cross` otherwise. A repetition between the two
orders of a single pair cannot appear, since such a pair is not an edge of $G$.

In [7]:
import os
import pandas as pd

path = "results/near_misses_X300000.csv"
if not os.path.exists(path):
    raise FileNotFoundError(
        "Run the exhaustive_sweep cell with bound 300_000 first.")

near = pd.read_csv(path)
print(f"{len(near)} near misses; collision kinds "
      f"{near['collision_kind'].value_counts().to_dict()}")
display(near.head(10))

89 near misses; collision kinds {'column': 89}


,p1,p2,p3,r12,r13,r21,r23,r31,r32,n_distinct,repeated_order,collision_kind
0,7,419,153269,19,38317,2,38317,3,11,5,38317,column
1,7,2741,28643,137,14321,3,14321,2,5,5,14321,column
2,7,2741,127343,137,63671,3,63671,2,5,5,63671,column
3,7,2741,258887,137,129443,3,129443,2,5,5,129443,column
4,7,4759,138917,61,34729,2,34729,3,13,5,34729,column
5,7,8269,253679,53,126839,3,126839,2,13,5,126839,column
6,7,14009,83243,17,41621,3,41621,2,103,5,41621,column
7,7,14009,128603,17,64301,3,64301,2,103,5,64301,column
8,7,24509,70139,557,35069,3,35069,2,11,5,35069,column
9,7,24509,127679,557,63839,3,63839,2,11,5,63839,column


In [8]:
example = near[(near.p1 == 11) & (near.p2 == 43) & (near.p3 == 16333)]
assert len(example) == 1, "Example 5.15 is missing from the export"
print("Example 5.15 of the article:")
print(example.to_string(index=False))

Example 5.15 of the article:
 p1  p2    p3  r12  r13  r21  r23  r31  r32  n_distinct  repeated_order collision_kind
 11  43 16333    7 1361    2 1361    5    3           5            1361         column


The repetition is always of the `column` shape: two orders modulo a common element.
That is the dominant obstruction, and it is forced whenever $p-1$ carries too few odd
prime factors -- for a safe prime $p-1 = 2q$ the two incoming orders have a single odd
prime to choose between and must collide.

---
## 4. Part B -- cyclotomic confinement

Part A certifies an interval; Part B certifies an unbounded region, using Theorem 5.10:
for primes $r$ and primes $q \notin \{p, r\}$,

$$\operatorname{ord}_q(p) = r \iff q \mid \Phi_r(p) = \frac{p^r - 1}{p - 1}.$$

Fixing $p$ and an exact order $r$ confines the modulus $q$ to the prime divisors of one
explicit integer, so factoring that integer yields the complete candidate list with no
upper bound on $q$. Every resulting pair is then completed: the remaining four orders are
computed and the six tested for being prime and pairwise distinct. What is decided is that
no aligned triple contains a prime $p < p_{\max}$ whose two outgoing orders are both at
most $r_{\max}$.

Two reported quantities bear on that claim. `skipped` lists any $r$ for which $\Phi_r(p)$
exceeded the digit cap and was not factored, and must be empty. `n_undecided` counts the
pairs left open; with `allow_hard_factoring=True` it is zero by construction, so the search
is repeated with the flag set to `False` to measure how many pairs the cheap branches alone
cannot settle.

In [9]:
confinement = confinement_search(p_max=100, r_max=23, digit_cap=50,
                                 allow_order_equal_to_base=True)

# The same search with the give-up path enabled, to measure how many pairs the
# cheap branches alone leave open; the run above settled those by factoring.
cautious = confinement_search(p_max=100, r_max=23, digit_cap=50,
                              allow_order_equal_to_base=True,
                              allow_hard_factoring=False, verbose=False)
print(f"pairs the cheap branches alone cannot settle: {cautious.n_undecided}")

  base prime p = 3
    r =   2: Phi_r(3) has   1 digits,   0 admissible prime divisors
    r =   3: Phi_r(3) has   2 digits,   1 admissible prime divisors
    r =   5: Phi_r(3) has   3 digits,   1 admissible prime divisors
    r =   7: Phi_r(3) has   4 digits,   1 admissible prime divisors
    r =  11: Phi_r(3) has   5 digits,   2 admissible prime divisors
    r =  13: Phi_r(3) has   6 digits,   1 admissible prime divisors
    r =  17: Phi_r(3) has   8 digits,   2 admissible prime divisors
    r =  19: Phi_r(3) has   9 digits,   2 admissible prime divisors
    r =  23: Phi_r(3) has  11 digits,   2 admissible prime divisors
  base prime p = 5
    r =   2: Phi_r(5) has   1 digits,   1 admissible prime divisors
    r =   3: Phi_r(5) has   2 digits,   1 admissible prime divisors
    r =   5: Phi_r(5) has   3 digits,   2 admissible prime divisors
    r =   7: Phi_r(5) has   5 digits,   1 admissible prime divisors
    r =  11: Phi_r(5) has   8 digits,   1 admissible prime divisors
    r =  1

---
## 5. Exploratory search (supports no claim in the article)

An unbounded heuristic hunt above the sweep bound. It certifies nothing; it is included
because it is the natural way to look for a positive example, and its negative outcome is
the reason none is reported.

An anchor pair $(p_1, p_2)$ with prime distinct mutual orders is fixed and a large $p_3$
is scanned. The orders $r_{31}, r_{32}$ depend on $p_3$ only through its residues modulo
the small primes $p_1, p_2$ and are read from a precomputed table in constant time; only
the survivors pay for the factorization of $p_3 - 1$ needed for $r_{13}, r_{23}$. Before
either step, a necessary condition removes most primes: if $p_3$ is the largest element of
an aligned triple then neither $p_1$ nor $p_2$ is congruent to $-1$ modulo $p_3$, since
that would force $p_1 = p_3 - 1$, which is even; so $r_{13}$ and $r_{23}$ are distinct odd
primes dividing $p_3 - 1$, and $p_3 - 1$ must carry at least two distinct odd prime
factors. Safe primes, for which $p_3 - 1 = 2q$, are excluded outright.

Anchors are ranked by an upper bound on the probability that a uniform $p_3$ passes the
cheap filter. The cell below only builds them; uncomment the scan to run it, and stop it
with the Colab stop button.

In [10]:
anchors = build_anchors(bound=6000, how_many=300)
for a in anchors[:5]:
    print(a)

# directed_search(anchors, start=300_000, block=5_000_000)

{'weight': 0.01904761904761905, 'p1': 11, 'p2': 43, 'r12': 7, 'r21': 2}
{'weight': 0.007974481658692184, 'p1': 7, 'p2': 419, 'r12': 19, 'r21': 2}
{'weight': 0.003787878787878788, 'p1': 13, 'p2': 419, 'r12': 11, 'r21': 3}
{'weight': 0.00326797385620915, 'p1': 13, 'p2': 103, 'r12': 17, 'r21': 2}
{'weight': 0.003003003003003003, 'p1': 7, 'p2': 223, 'r12': 37, 'r21': 2}


---
## 6. Summary

`results/runs.jsonl` accumulates one JSON line per completed run, so every number quoted
in the article can be traced back to a specific execution with its parameters and timing.

In [11]:
import json

for line in open("results/runs.jsonl"):
    record = json.loads(line)
    print(record["part"], record["timestamp"],
          {k: v for k, v in record.items()
           if k.startswith("n_") or k in ("bound", "p_max", "r_max", "seconds")})

A 2026-08-07T01:57:35 {'bound': 3000, 'n_odd_primes': 429, 'n_ordered_pairs': 183612, 'n_prime_order_pairs': 21680, 'n_edges': 1296, 'n_triangles': 345, 'n_aligned': 0, 'n_five_distinct': 0, 'seconds': 0.2}
A 2026-08-07T02:06:04 {'bound': 300000, 'n_odd_primes': 25996, 'n_ordered_pairs': 675766020, 'n_prime_order_pairs': 42465610, 'n_edges': 1237384, 'n_triangles': 8877957, 'n_aligned': 0, 'n_five_distinct': 89, 'seconds': 475.9}
B 2026-08-07T02:08:25 {'p_max': 100, 'r_max': 23, 'n_base_primes': 24, 'n_candidate_pairs': 3949, 'n_aligned': 0, 'n_undecided': 0, 'seconds': 17.4}
B 2026-08-07T02:08:26 {'p_max': 100, 'r_max': 23, 'n_base_primes': 24, 'n_candidate_pairs': 3949, 'n_aligned': 0, 'n_undecided': 1, 'seconds': 0.3}
